# F_S3 — Shape Classification Pipeline

Implements the full **global-to-shape decision hierarchy**:

1. **Gate 1** — MIC → global relationship strength (strong / intermediate / none)
2. **Gate 2** — |Pearson| & |Spearman| → Simple vs Complex
3. **Simple path** — power-law fit → Linear / Concave / Convex / S-shaped / Threshold
4. **Complex path** — Branching / Threshold / Turning-point → subtypes

Reads from `F/output/S1_parameterized/` and `F/output/S2_parameterized/`.

In [15]:
from __future__ import annotations

import json
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.optimize import curve_fit
from statsmodels.nonparametric.smoothers_lowess import lowess

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(it, **kw):
        return it

warnings.filterwarnings('ignore')

def locate_repo_root() -> Path:
    for root in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (root / 'D').is_dir() and (root / 'E').is_dir():
            return root
    raise FileNotFoundError('Could not locate repo root.')

REPO_ROOT = locate_repo_root()
S1_DIR = REPO_ROOT / 'F' / 'output' / 'S1_parameterized'
S2_DIR = REPO_ROOT / 'F' / 'output' / 'S2_metrics'
OUT_DIR = REPO_ROOT / 'F' / 'output' / 'S3_shape_classification'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Gate 1 thresholds ──
MIC_STRONG = 0.8
MIC_INTERMEDIATE = 0.6

# ── Gate 2 thresholds ──
SIMPLE_CORR_THRESHOLD = 0.7

# ── Simple path thresholds ──
R2_POWER_THRESHOLD = 0.85          # min R² for adequate power-law fit
LINEAR_B_TOLERANCE = 0.05          # |b - 1| <= this → Linear
LOWESS_FRAC = 0.25                 # LOWESS bandwidth
S_SHAPE_SHARPNESS_MAX = 6.0        # max deriv / median deriv below this → S-shaped (not threshold)
THRESHOLD_SHARPNESS_MIN = 6.0      # max deriv / median deriv above this → Threshold

# ── Complex path thresholds ──
BRANCHING_GAP_RATIO = 0.25         # gap / y_range in a bin to count as branching evidence
BRANCHING_MIN_BINS = 3             # min bins showing gaps
COMPLEX_THRESHOLD_DERIV_RATIO = 5.0  # derivative spike ratio for threshold detection

print(f'Gate 1: strong MIC >= {MIC_STRONG}, intermediate MIC >= {MIC_INTERMEDIATE}')
print(f'Gate 2: Simple when |r| >= {SIMPLE_CORR_THRESHOLD} AND |ρ| >= {SIMPLE_CORR_THRESHOLD}')
print(f'Simple path: power-law R² >= {R2_POWER_THRESHOLD}, linear |b−1| <= {LINEAR_B_TOLERANCE}')
print(f'Complex path: branching gap ratio >= {BRANCHING_GAP_RATIO} in >= {BRANCHING_MIN_BINS} bins')

Gate 1: strong MIC >= 0.8, intermediate MIC >= 0.6
Gate 2: Simple when |r| >= 0.7 AND |ρ| >= 0.7
Simple path: power-law R² >= 0.85, linear |b−1| <= 0.05
Complex path: branching gap ratio >= 0.25 in >= 3 bins


## Family catalogue

Ground-truth properties for each signal family, used for validation at the end.

In [16]:
# Family catalogue: ground-truth properties for each signal family.
# F17 (Quadratic peak) was removed in F; F18 is standalone.
FAMILY_CATALOGUE = {
    'F01': {'name': 'Linear positive',           'direction': 'positive',     'linearity': 'linear',              'monotonicity': 'monotonic',     'curvature': 'N/A',    'special_shape': 'none',                  'tp': 0},
    'F02': {'name': 'Linear negative',           'direction': 'negative',     'linearity': 'linear',              'monotonicity': 'monotonic',     'curvature': 'N/A',    'special_shape': 'none',                  'tp': 0},
    'F03': {'name': 'Power convex positive',     'direction': 'positive',     'linearity': 'nonlinear',           'monotonicity': 'monotonic',     'curvature': 'convex', 'special_shape': 'none',                  'tp': 0},
    'F04': {'name': 'Power convex negative',     'direction': 'negative',     'linearity': 'nonlinear',           'monotonicity': 'monotonic',     'curvature': 'concave','special_shape': 'none',                  'tp': 0},
    'F05': {'name': 'Power concave positive',    'direction': 'positive',     'linearity': 'nonlinear',           'monotonicity': 'monotonic',     'curvature': 'concave','special_shape': 'none',                  'tp': 0},
    'F06': {'name': 'Power concave negative',    'direction': 'negative',     'linearity': 'nonlinear',           'monotonicity': 'monotonic',     'curvature': 'convex', 'special_shape': 'none',                  'tp': 0},
    'F07': {'name': 'Saturation positive',       'direction': 'positive',     'linearity': 'nonlinear',           'monotonicity': 'monotonic',     'curvature': 'concave','special_shape': 'saturation',            'tp': 0},
    'F08': {'name': 'Saturation negative',       'direction': 'negative',     'linearity': 'nonlinear',           'monotonicity': 'monotonic',     'curvature': 'convex', 'special_shape': 'saturation',            'tp': 0},
    'F09': {'name': 'Log positive',              'direction': 'positive',     'linearity': 'nonlinear',           'monotonicity': 'monotonic',     'curvature': 'concave','special_shape': 'none',                  'tp': 0},
    'F10': {'name': 'Log negative',              'direction': 'negative',     'linearity': 'nonlinear',           'monotonicity': 'monotonic',     'curvature': 'convex', 'special_shape': 'none',                  'tp': 0},
    'F11': {'name': 'Exponential positive',      'direction': 'positive',     'linearity': 'nonlinear',           'monotonicity': 'monotonic',     'curvature': 'convex', 'special_shape': 'none',                  'tp': 0},
    'F12': {'name': 'Exponential negative',      'direction': 'negative',     'linearity': 'nonlinear',           'monotonicity': 'monotonic',     'curvature': 'concave','special_shape': 'none',                  'tp': 0},
    'F13': {'name': 'S-curve positive',          'direction': 'positive',     'linearity': 'nonlinear',           'monotonicity': 'monotonic',     'curvature': 'mixed',  'special_shape': 'S-curve',               'tp': 0},
    'F14': {'name': 'S-curve negative',          'direction': 'negative',     'linearity': 'nonlinear',           'monotonicity': 'monotonic',     'curvature': 'mixed',  'special_shape': 'S-curve',               'tp': 0},
    'F15': {'name': 'Threshold positive',        'direction': 'positive',     'linearity': 'nonlinear',           'monotonicity': 'monotonic',     'curvature': 'N/A',    'special_shape': 'threshold',             'tp': 0},
    'F16': {'name': 'Threshold negative',        'direction': 'negative',     'linearity': 'nonlinear',           'monotonicity': 'monotonic',     'curvature': 'N/A',    'special_shape': 'threshold',             'tp': 0},
    'F18': {'name': 'Quadratic valley (U)',       'direction': 'none/local',   'linearity': 'nonlinear',           'monotonicity': 'non-monotonic', 'curvature': 'convex', 'special_shape': 'valley',                'tp': 1},
    'F19': {'name': 'Spike',                     'direction': 'none/local',   'linearity': 'nonlinear',           'monotonicity': 'non-monotonic', 'curvature': 'N/A',    'special_shape': 'spike',                 'tp': 1},
    'F20': {'name': 'Inverted Spike',            'direction': 'none/local',   'linearity': 'nonlinear',           'monotonicity': 'non-monotonic', 'curvature': 'N/A',    'special_shape': 'inverted spike',        'tp': 1},
    'F21': {'name': 'Cubic',                     'direction': 'none/mixed',   'linearity': 'nonlinear',           'monotonicity': 'non-monotonic', 'curvature': 'mixed',  'special_shape': 'cubic',                 'tp': 2},
    'F22': {'name': 'Oscillation',               'direction': 'none/mixed',   'linearity': 'nonlinear',           'monotonicity': 'non-monotonic', 'curvature': 'mixed',  'special_shape': 'oscillation',           'tp': 'multiple'},
    'F23': {'name': 'Two Lines',                 'direction': 'positive',     'linearity': 'multi-branch linear', 'monotonicity': 'multi-branch',  'curvature': 'N/A',    'special_shape': 'two lines',             'tp': 0},
    'F24': {'name': 'Line and Parabola',         'direction': 'none/local',   'linearity': 'multi-branch',        'monotonicity': 'multi-branch',  'curvature': 'mixed',  'special_shape': 'line + parabola',       'tp': 1},
    'F25': {'name': 'Multi-regime threshold',    'direction': 'positive',     'linearity': 'multi-branch',        'monotonicity': 'multi-branch',  'curvature': 'convex', 'special_shape': 'overlapping J',         'tp': 'multiple'},
    'F26': {'name': 'Windowed threshold',        'direction': 'positive',     'linearity': 'nonlinear',           'monotonicity': 'non-monotonic', 'curvature': 'convex', 'special_shape': 'windowed J',            'tp': 'multiple'},
    'Null': {'name': 'No relationship',          'direction': 'N/A',          'linearity': 'N/A',                 'monotonicity': 'N/A',           'curvature': 'N/A',    'special_shape': 'none',                  'tp': 'N/A'},
}

# Build the expected shape_label mapping from the catalogue.
# This is the ground-truth: what shape should the classifier assign at high SNR?
EXPECTED_SHAPE = {}
for fid, props in FAMILY_CATALOGUE.items():
    if fid == 'Null':
        EXPECTED_SHAPE[fid] = 'no_global'
        continue
    d = props['direction']
    mono = props['monotonicity']
    lin = props['linearity']
    curv = props['curvature']
    shape = props['special_shape']
    tp = props['tp']

    # Simple monotonic families → gate2 = Simple
    if mono == 'monotonic':
        if lin == 'linear':
            EXPECTED_SHAPE[fid] = f'{d}_line'
        elif shape == 'S-curve':
            EXPECTED_SHAPE[fid] = f'{d}_s_shaped'
        elif shape == 'threshold':
            EXPECTED_SHAPE[fid] = f'{d}_threshold'
        elif curv == 'concave':
            EXPECTED_SHAPE[fid] = f'{d}_concave'
        elif curv == 'convex':
            EXPECTED_SHAPE[fid] = f'{d}_convex'
        else:
            EXPECTED_SHAPE[fid] = 'simple_other'
    # Non-monotonic / multi-branch → gate2 = Complex
    elif mono == 'non-monotonic':
        if tp == 1:
            EXPECTED_SHAPE[fid] = 'u_inv_u_spike'
        elif tp == 2:
            EXPECTED_SHAPE[fid] = 'complex_cubic'
        elif tp == 'multiple' and 'oscillat' in shape:
            EXPECTED_SHAPE[fid] = 'oscillatory'
        elif tp == 'multiple' and 'windowed' in shape:
            EXPECTED_SHAPE[fid] = 'multi_step_threshold'
        else:
            EXPECTED_SHAPE[fid] = 'complex_uncertain'
    elif mono == 'multi-branch':
        if 'two lines' in shape:
            EXPECTED_SHAPE[fid] = '2_way_branching'
        elif 'parabola' in shape:
            EXPECTED_SHAPE[fid] = '2_way_branching'
        elif 'overlapping' in shape or 'J' in shape:
            EXPECTED_SHAPE[fid] = 'n_way_branching'
        else:
            EXPECTED_SHAPE[fid] = 'complex_uncertain'

catalogue_df = pd.DataFrame.from_dict(FAMILY_CATALOGUE, orient='index').reset_index(names='family_id')
catalogue_df['expected_shape'] = catalogue_df['family_id'].map(EXPECTED_SHAPE)
print(f'{len(FAMILY_CATALOGUE)} families loaded')
display(catalogue_df[['family_id', 'name', 'monotonicity', 'curvature', 'special_shape', 'expected_shape']])

26 families loaded


,family_id,name,monotonicity,curvature,special_shape,expected_shape
0,F01,Linear positive,monotonic,N/A,none,positive_line
1,F02,Linear negative,monotonic,N/A,none,negative_line
2,F03,Power convex positive,monotonic,convex,none,positive_convex
3,F04,Power convex negative,monotonic,concave,none,negative_concave
4,F05,Power concave positive,monotonic,concave,none,positive_concave
5,F06,Power concave negative,monotonic,convex,none,negative_convex
6,F07,Saturation positive,monotonic,concave,saturation,positive_concave
7,F08,Saturation negative,monotonic,convex,saturation,negative_convex
8,F09,Log positive,monotonic,concave,none,positive_concave
9,F10,Log negative,monotonic,convex,none,negative_convex


## Load data

In [17]:
cases_df = pd.read_csv(S1_DIR / 'cases.csv', low_memory=False)
metrics_df = pd.read_parquet(S2_DIR / 'metrics_full.parquet')
pts = np.load(S1_DIR / 'scatter_points.npz')
x_all = pts['x']
y_all = pts['y']
del pts

data = cases_df.merge(metrics_df, on='case_id', how='inner', validate='one_to_one')
data['is_null'] = data['family_id'].eq('Null')

print(f'Loaded {len(data):,} cases × {x_all.shape[1]} points')
print(f'  Signal: {(~data["is_null"]).sum():,}   Null: {data["is_null"].sum():,}')
print(f'  MIC range: [{data["MIC"].min():.3f}, {data["MIC"].max():.3f}]')

Loaded 335,960 cases × 500 points
  Signal: 326,880   Null: 9,080
  MIC range: [0.122, 1.000]


## Gate 1 — Global relationship (MIC)

- **Strong**: MIC ≥ 0.8 → proceed to Gate 2
- **Intermediate**: 0.6 ≤ MIC < 0.8 → classification stops here
- **No global**: MIC < 0.6 → no relationship detected

In [18]:
def assign_gate1(mic):
    if mic >= MIC_STRONG:
        return 'strong_global'
    elif mic >= MIC_INTERMEDIATE:
        return 'intermediate_global'
    else:
        return 'no_global'

data['gate1'] = data['MIC'].apply(assign_gate1)

_g1 = data.groupby(['family_id', 'gate1']).size().unstack(fill_value=0)
_g1['total'] = _g1.sum(axis=1)
for col in ['strong_global', 'intermediate_global', 'no_global']:
    if col in _g1.columns:
        _g1[f'{col}_%'] = (_g1[col] / _g1['total'] * 100).round(1)
_g1 = _g1.sort_index()
_g1.insert(0, 'family_name', _g1.index.map(lambda f: FAMILY_CATALOGUE.get(f, {}).get('name', '')))
col_order = ['family_name', 'total']
for col in ['strong_global', 'intermediate_global', 'no_global']:
    if col in _g1.columns:
        col_order += [col, f'{col}_%']
_g1 = _g1[col_order]
print('=== Gate 1: Per-family MIC classification ===')
display(_g1)
print(f'\nOverall: {(data["gate1"]=="strong_global").sum():,} strong / '
      f'{(data["gate1"]=="intermediate_global").sum():,} intermediate / '
      f'{(data["gate1"]=="no_global").sum():,} no_global '
      f'out of {len(data):,} total')

## Gate 2 — Simple or Complex

Applied only to **strong global** cases (MIC ≥ 0.8).

- **Simple**: |Pearson r| ≥ 0.7 AND |Spearman ρ| ≥ 0.7
- **Complex**: otherwise

In [19]:
mask_strong = data['gate1'] == 'strong_global'

data['gate2'] = None
data.loc[mask_strong, 'gate2'] = np.where(
    (data.loc[mask_strong, 'pearson_r'].abs() >= SIMPLE_CORR_THRESHOLD)
    & (data.loc[mask_strong, 'spearman_rho'].abs() >= SIMPLE_CORR_THRESHOLD),
    'Simple', 'Complex'
)

_g2_data = data.loc[mask_strong]
_g2 = _g2_data.groupby(['family_id', 'gate2']).size().unstack(fill_value=0)
_g2['total_strong'] = _g2.sum(axis=1)
for col in ['Simple', 'Complex']:
    if col in _g2.columns:
        _g2[f'{col}_%'] = (_g2[col] / _g2['total_strong'] * 100).round(1)
_g2 = _g2.sort_index()
_g2.insert(0, 'family_name', _g2.index.map(lambda f: FAMILY_CATALOGUE.get(f, {}).get('name', '')))
col_order = ['family_name', 'total_strong']
for col in ['Simple', 'Complex']:
    if col in _g2.columns:
        col_order += [col, f'{col}_%']
_g2 = _g2[col_order]
print('=== Gate 2: Per-family Simple vs Complex (strong_global only) ===')
display(_g2)
print(f'\nOverall strong: {_g2_data["gate2"].eq("Simple").sum():,} Simple / '
      f'{_g2_data["gate2"].eq("Complex").sum():,} Complex '
      f'out of {len(_g2_data):,} strong_global')

## Simple path — shape classification

Pipeline for cases classified as **Simple** (high correlation):

1. **Power-law fit**: y = a·x^b + c
   - Adequate (R² ≥ threshold) → check linearity and curvature
   - Linear: |b − 1| ≤ 0.05 → **Positive/Negative Line**
   - Curvature sign a·b·(b−1): **Concave** or **Convex** + direction
2. **Cubic polynomial fit**: y = a₃x³ + a₂x² + a₁x + a₀
   - Two interior turning points (roots of f′ in [0.2, 0.8]) → **Cubic/two-turning-point**
3. **LOWESS shape analysis**:
   - Monotonic + inflection → **S-shaped** + direction
   - Sharp derivative spike → **Threshold** + direction
   - Otherwise → **Other/Uncertain**

In [20]:
def _power_law(x, a, b, c):
    return a * np.power(np.maximum(x, 1e-10), b) + c


def _fit_power_law(x, y):
    """Returns (a, b, c, r2) or None."""
    valid = np.isfinite(x) & np.isfinite(y) & (x > 1e-6)
    if valid.sum() < 10:
        return None
    x, y = x[valid], y[valid]
    slope = np.polyfit(x, y, 1)[0]
    a0 = slope
    for b0 in [1.0, 0.5, 2.0, 0.3, 3.0]:
        try:
            popt, _ = curve_fit(
                _power_law, x, y, p0=[a0, b0, np.median(y)],
                maxfev=5000,
                bounds=([-np.inf, 0.05, -np.inf], [np.inf, 8.0, np.inf]),
            )
            y_pred = _power_law(x, *popt)
            ss_res = np.sum((y - y_pred) ** 2)
            ss_tot = np.sum((y - y.mean()) ** 2)
            r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0
            if r2 >= 0:
                return (*popt, r2)
        except (RuntimeError, ValueError):
            continue
    return None


def _fit_cubic(x, y):
    """Returns (coeffs, interior_turning_points, r2) or None."""
    valid = np.isfinite(x) & np.isfinite(y)
    if valid.sum() < 10:
        return None
    x, y = x[valid], y[valid]
    try:
        coeffs = np.polyfit(x, y, 3)
        y_pred = np.polyval(coeffs, x)
        ss_res = np.sum((y - y_pred) ** 2)
        ss_tot = np.sum((y - y.mean()) ** 2)
        r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0
        deriv = np.polyder(coeffs)
        roots = np.roots(deriv)
        real_roots = roots[np.isreal(roots)].real
        interior = real_roots[(real_roots >= 0.2) & (real_roots <= 0.8)]
        return coeffs, interior, r2
    except (np.RankWarning, np.linalg.LinAlgError):
        return None


def _lowess_shape_analysis(x, y, frac=LOWESS_FRAC):
    """Analyze LOWESS curve for S-shape or threshold evidence.
    Returns (is_monotonic, has_inflection, sharpness_ratio, overall_direction).
    """
    valid = np.isfinite(x) & np.isfinite(y)
    if valid.sum() < 20:
        return False, False, 0.0, 0.0
    x, y = x[valid], y[valid]
    order = np.argsort(x)
    x, y = x[order], y[order]
    fitted = lowess(y, x, frac=frac, return_sorted=True)
    x_fit, y_fit = fitted[:, 0], fitted[:, 1]
    if len(x_fit) < 5:
        return False, False, 0.0, 0.0

    dx = np.diff(x_fit)
    dy = np.diff(y_fit)
    ok = dx > 1e-10
    if ok.sum() < 3:
        return False, False, 0.0, 0.0
    deriv = dy[ok] / dx[ok]

    # Monotonicity: count significant sign changes in derivative
    tol = 0.05 * np.max(np.abs(deriv)) if np.max(np.abs(deriv)) > 0 else 1e-10
    signs = np.zeros_like(deriv, dtype=int)
    signs[deriv > tol] = 1
    signs[deriv < -tol] = -1
    nz = signs[signs != 0]
    sign_changes = int(np.sum(nz[1:] != nz[:-1])) if len(nz) >= 2 else 0
    is_monotonic = sign_changes == 0

    # Inflection: sign change in second derivative
    dderiv = np.diff(deriv)
    tol2 = 0.05 * np.max(np.abs(dderiv)) if len(dderiv) > 0 and np.max(np.abs(dderiv)) > 0 else 1e-10
    signs2 = np.zeros_like(dderiv, dtype=int)
    signs2[dderiv > tol2] = 1
    signs2[dderiv < -tol2] = -1
    nz2 = signs2[signs2 != 0]
    has_inflection = len(nz2) >= 2 and np.any(nz2[1:] != nz2[:-1])

    # Sharpness: max|deriv| / median|deriv|
    abs_deriv = np.abs(deriv)
    median_d = np.median(abs_deriv)
    sharpness = float(np.max(abs_deriv) / median_d) if median_d > 1e-10 else 0.0

    # Overall direction
    overall_direction = float(y_fit[-1] - y_fit[0])

    return is_monotonic, has_inflection, sharpness, overall_direction


def classify_simple_path(x, y):
    """Full Simple path classification. Returns (shape_label, details_dict)."""
    # Step 1: Power-law fit
    pw = _fit_power_law(x, y)
    if pw is not None:
        a, b, c, r2 = pw
        if r2 >= R2_POWER_THRESHOLD:
            ab = a * b
            direction = 'positive' if ab > 0 else 'negative'
            details = {'fit': 'power_law', 'a': a, 'b': b, 'c': c, 'r2': r2}
            if abs(b - 1) <= LINEAR_B_TOLERANCE:
                return f'{direction}_line', details
            curvature = a * b * (b - 1)
            if curvature < 0:
                return f'{direction}_concave', details
            else:
                return f'{direction}_convex', details

    # Step 2: Cubic polynomial fit
    cb = _fit_cubic(x, y)
    if cb is not None:
        coeffs, interior_tp, r2_cubic = cb
        if len(interior_tp) >= 2:
            return 'cubic_two_turning_point', {
                'fit': 'cubic', 'coeffs': coeffs.tolist(),
                'interior_turning_points': interior_tp.tolist(), 'r2': r2_cubic,
            }

    # Step 3: LOWESS shape analysis → S-shaped or Threshold
    is_mono, has_infl, sharpness, direction_val = _lowess_shape_analysis(x, y)
    direction = 'positive' if direction_val >= 0 else 'negative'
    details = {
        'fit': 'lowess', 'is_monotonic': is_mono,
        'has_inflection': has_infl, 'sharpness': sharpness,
    }

    if is_mono and sharpness >= THRESHOLD_SHARPNESS_MIN:
        return f'{direction}_threshold', details
    if is_mono and has_infl:
        return f'{direction}_s_shaped', details
    if sharpness >= THRESHOLD_SHARPNESS_MIN:
        return f'{direction}_threshold', details

    return 'simple_other', details

print('Simple path helpers defined.')

Simple path helpers defined.


## Run full classification

In [21]:
import time

shape_labels = np.full(len(data), '', dtype=object)
shape_details = np.full(len(data), '', dtype=object)

mask_intermediate = data['gate1'] == 'intermediate_global'
mask_no_global = data['gate1'] == 'no_global'
shape_labels[mask_intermediate.values] = 'intermediate_global'
shape_labels[mask_no_global.values] = 'no_global'
shape_details[mask_intermediate.values] = '{}'
shape_details[mask_no_global.values] = '{}'

mask_simple = (data['gate2'] == 'Simple').values
mask_complex = (data['gate2'] == 'Complex').values
idx_simple = np.where(mask_simple)[0]

print(f'To classify: {len(idx_simple):,} Simple')
print(f'Complex (not sub-classified): {mask_complex.sum():,}')
print(f'Skipped: {mask_intermediate.sum():,} intermediate + {mask_no_global.sum():,} no_global')

t0 = time.time()
for i in tqdm(idx_simple, desc='Simple path'):
    x = x_all[i].astype(np.float64)
    y = y_all[i].astype(np.float64)
    label, details = classify_simple_path(x, y)
    shape_labels[i] = label
    shape_details[i] = json.dumps(details, default=str)
print(f'Simple path: {len(idx_simple):,} cases in {time.time() - t0:.1f}s')

shape_labels[mask_complex] = 'complex'
shape_details[mask_complex] = '{}'

data['shape_label'] = shape_labels
data['shape_details'] = shape_details

# Aggregate into broad shape categories
SHAPE_TO_BROAD = {
    'positive_line': 'line', 'negative_line': 'line',
    'positive_concave': 'concave', 'negative_concave': 'concave',
    'positive_convex': 'convex', 'negative_convex': 'convex',
    'cubic_two_turning_point': 'cubic',
    'positive_s_shaped': 'S-curve', 'negative_s_shaped': 'S-curve',
    'positive_threshold': 'threshold', 'negative_threshold': 'threshold',
    'simple_other': 'S-curve/threshold',
}
BROAD_ORDER = ['line', 'concave', 'convex', 'cubic', 'S-curve', 'threshold']

_simple = data.loc[mask_simple].copy()
_simple['broad_shape'] = _simple['shape_label'].map(SHAPE_TO_BROAD).fillna('other')
_s_ct = _simple.groupby(['family_id', 'broad_shape']).size().unstack(fill_value=0)
_s_ct = _s_ct.reindex(columns=BROAD_ORDER, fill_value=0)
_s_ct.insert(0, 'total', _s_ct.sum(axis=1))
_s_ct.insert(0, 'family_name', _s_ct.index.map(lambda f: FAMILY_CATALOGUE.get(f, {}).get('name', '')))
print('\n=== Simple path: Per-family shape classification ===')
display(_s_ct)

## Save results

In [22]:
output_cols = [
    'case_id', 'family_id', 'family_name', 'canonical_family',
    'strength_name', 'snr', 'snr_float', 'replicate',
    'MIC', 'pearson_r', 'spearman_rho',
    'gate1', 'gate2', 'shape_label', 'shape_details',
]
if 'is_representative' in data.columns:
    output_cols.insert(4, 'is_representative')

result = data[output_cols].copy()
result.to_parquet(OUT_DIR / 'shape_classification.parquet', index=False)
result.to_csv(OUT_DIR / 'shape_classification.csv', index=False)

print(f'Saved {len(result):,} rows to {OUT_DIR.relative_to(REPO_ROOT)}/')
print(f'  shape_classification.parquet')
print(f'  shape_classification.csv')

Saved 335,960 rows to F/output/S3_shape_classification/
  shape_classification.parquet
  shape_classification.csv


## Summary & validation

Compare predicted shape labels against the known `family_id` ground truth.

In [23]:
# Overall shape label distribution
print('=== Shape label distribution ===')
print(data['shape_label'].value_counts().to_string())
print()

# Per-family accuracy at high SNR (where classification should be reliable)
high_snr = data[data['snr_float'] >= 10].copy()
high_snr['expected'] = high_snr['family_id'].map(EXPECTED_SHAPE)
high_snr['correct'] = high_snr['shape_label'] == high_snr['expected']

print('=== Per-family accuracy (SNR >= 10) ===')
accuracy = high_snr.groupby('family_id').agg(
    n=('correct', 'size'),
    correct=('correct', 'sum'),
    accuracy=('correct', 'mean'),
    top_predicted=('shape_label', lambda s: s.value_counts().index[0]),
    expected=('expected', 'first'),
).sort_values('accuracy')
print(accuracy.to_string())
print(f'\nOverall accuracy (SNR >= 10): {high_snr["correct"].mean():.1%}')

# Families with low accuracy → likely need threshold tuning
low_acc = accuracy[accuracy['accuracy'] < 0.5]
if len(low_acc) > 0:
    print(f'\n⚠ {len(low_acc)} families below 50% accuracy:')
    for fid, row in low_acc.iterrows():
        print(f'  {fid}: expected={row["expected"]}, got={row["top_predicted"]} ({row["accuracy"]:.0%})')

=== Shape label distribution ===
shape_label
no_global                  210792
intermediate_global         32706
complex                     26276
positive_line               17571
negative_line               14997
positive_convex             12967
negative_concave            12020
positive_concave             6470
positive_s_shaped             788
negative_s_shaped             595
simple_other                  355
negative_convex               293
negative_threshold             78
cubic_two_turning_point        31
positive_threshold             21

=== Per-family accuracy (SNR >= 10) ===
               n  correct  accuracy        top_predicted              expected
family_id                                                                     
F05        10440        0  0.000000        negative_line      positive_concave
F07        10440        0  0.000000  intermediate_global      positive_concave
F13        10440        0  0.000000      positive_convex     positive_s_shaped
F15      

In [24]:
# Cross-tabulation: family × shape_label for high SNR
print('=== Family × Shape Label (SNR >= 10, strong_global only) ===')
ct = high_snr[high_snr['gate1'] == 'strong_global'].groupby(
    ['family_id', 'shape_label']
).size().unstack(fill_value=0)
display(ct)

=== Family × Shape Label (SNR >= 10, strong_global only) ===


shape_label,complex,negative_concave,negative_convex,negative_line,positive_concave,positive_convex,positive_line
family_id,,,,,,,
F01,0,0,0,0,125,140,10175
F03,0,42,26,3337,75,80,6534
F05,0,140,128,10136,0,0,0
F07,758,0,1,1003,0,1174,0
F13,0,0,0,0,0,10440,0
F15,2172,8264,0,0,0,0,0
F18,4318,2689,0,0,2985,0,0
F21,3479,0,0,0,3099,0,0
F22,10440,0,0,0,0,0,0
